In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import joblib
import matplotlib.pyplot as plt

# Настройки
SYMBOL = 'BTC-USD'
START_DATE = '2020-01-01'
LOOK_BACK = 60 # Прозорец за LSTM

# 1. Зареждане (Raw Data)
print("📥 Зареждане на данни...")
df = yf.download(SYMBOL, start=START_DATE)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# --- КРИТЕРИЙ 1: Липсващи стойности ---
# При крипто пазарите (24/7) липсващи данни обикновено са грешка на API-то.
# Стратегия: Forward Fill (попълване с предходната стойност), защото цената не може да е 0.
if df.isnull().sum().sum() > 0:
    print("⚠️ Открити липсващи стойности. Прилагане на Forward Fill.")
    df.fillna(method='ffill', inplace=True)
else:
    print("✅ Няма липсващи стойности.")

# --- КРИТЕРИЙ 7: Създаване на нови признаци (Feature Engineering) ---
# Създаваме признаците ПРЕДИ скалирането
df['Returns'] = df['Close'].pct_change()
df['SMA_7'] = df['Close'].rolling(window=7).mean()
df['Volatility'] = df['Close'].rolling(window=7).std()

# --- КРИТЕРИЙ 2: Обработка на Outliers ---
# Финансовите данни имат "дебели опашки". Не трием редове, но ограничаваме екстремни грешки.
# Ако дневната промяна е над 50% (невъзможно за BTC), я "отрязваме" (clipping).
df['Returns'] = df['Returns'].clip(lower=-0.5, upper=0.5)

# Премахване на NaN от feature engineering-а (първите 7 реда)
df.dropna(inplace=True)

# --- КРИТЕРИЙ 5: Разделяне на данните (Time Series Split) ---
# ВАЖНО: Не използваме random split, за да няма Data Leakage!
train_size = int(len(df) * 0.8)
val_size = int(len(df) * 0.1)

train_data = df.iloc[:train_size]
val_data = df.iloc[train_size:train_size+val_size]
test_data = df.iloc[train_size+val_size:]

print(f"📊 Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# --- КРИТЕРИЙ 3: Нормализация и Мащабиране ---
# Обучаваме скалера САМО върху Train данните!
scaler = MinMaxScaler(feature_range=(0, 1))

# Избираме колоните за модела
features = ['Close', 'SMA_7', 'Volatility']
train_scaled = scaler.fit_transform(train_data[features])

# Трансформираме Validation и Test със скалера от Train
val_scaled = scaler.transform(val_data[features])
test_scaled = scaler.transform(test_data[features])

# Запазване на скалера за Web приложението
joblib.dump(scaler, 'scaler.gz')
print("✅ Scaler записан успешно.")

# --- КРИТЕРИЙ 6: Възпроизводим Pipeline (Sequencing) ---
def create_sequences(dataset, look_back=60):
    X, y = [], []
    for i in range(look_back, len(dataset)):
        # Взимаме всички features за прозореца
        X.append(dataset[i-look_back:i])
        # Целта е само 'Close' (колона 0)
        y.append(dataset[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, LOOK_BACK)
X_val, y_val = create_sequences(val_scaled, LOOK_BACK)
X_test, y_test = create_sequences(test_scaled, LOOK_BACK)

# --- КРИТЕРИЙ 8: Валидиране на признаците ---
print(f"Feature Check - Max Value in Train: {X_train.max()}") # Трябва да е <= 1
print(f"Feature Check - Min Value in Train: {X_train.min()}") # Трябва да е >= 0
print(f"Final Input Shape: {X_train.shape}") # (Samples, 60, 3)

# Запазване на обработените данни (по желание)
np.save('../ml_engine/X_train.npy', X_train)
np.save('../ml_engine/y_train.npy', y_train)

📥 Зареждане на данни...


[*********************100%***********************]  1 of 1 completed

✅ Няма липсващи стойности.
📊 Train: 1783, Val: 222, Test: 224
✅ Scaler записан успешно.
Feature Check - Max Value in Train: 1.0000000000000002
Feature Check - Min Value in Train: 0.0
Final Input Shape: (1723, 60, 3)
